# Clasificación de Noticias

## Data Preparation

### TF-IDF

En esta sección preparamos los datos para entrenar modelos de clasificación. Cargamos las representaciones TF-IDF generadas previamente que capturan la importancia de cada término en los documentos. Estos vectores servirán como features (X) para predecir los topics de las noticias financieras (y). Dividimos el dataset en conjuntos de entrenamiento (80%), validación (10%) y test (10%) para evaluar el rendimiento de los modelos.

In [2]:
import pandas as pd
import os

tfidf_data = pd.read_parquet('../Representacion_del_lenguaje/datasPost_prepro/tfidf_embeddings.parquet')
tfidf_cols = [c for c in tfidf_data.columns if c.startswith("tfidf_")]
X = tfidf_data[tfidf_cols].values
print(X.shape)


(5160, 5000)


In [3]:
import numpy as np

df = pd.read_csv("../../data/definitivos/INDEX_ALL_scrapped_filtrado.csv")
y = df["topic"].values

print("Total de textos:", y.size)
print("\nDistribución de textos por topic:")
print(df["topic"].value_counts().sort_index())

Total de textos: 5160

Distribución de textos por topic:
topic
Business Growth and Cloud Infrastructure in the AI Industry    1539
Financial and Market News and Corporate Sales                  2257
Informal / Conversational Lenguaje                              152
Quantum Computing and Military Technology                       101
Stock Market and Trading                                       1111
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Textos (variable predictoria):\n", "Training", X_train.shape, "Validation", X_val.shape, "Test", X_test.shape)
print("Topics (variable a predecir):\n", "Training", y_train.size, "Validation", y_val.size, "Test", y_test.size)

Textos (variable predictoria):
 Training (4128, 5000) Validation (516, 5000) Test (516, 5000)
Topics (variable a predecir):
 Training 4128 Validation 516 Test 516


### Emb. No Contextuales

In [7]:
import pandas as pd
import os

no_context = pd.read_pickle('../Representacion_del_lenguaje/datasPost_prepro/tfidf_vectorizer.pkl')
no_context



TfidfVectorizer(max_df=0.8, max_features=5000, min_df=2, ngram_range=(1, 2))

## *Multiclass Logistic Regression*

Vamos primero a implementar una Regresión Logística Multinomial, un modelo clásico de machine learning que es eficiente y efectivo para clasificación de texto. Utilizamos el solver 'lbfgs' que es apropiado para problemas multiclase. Este modelo baseline nos permitirá establecer un punto de referencia para comparar con modelos más complejos posteriormente.

In [5]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(multi_class='multinomial',
                             solver='lbfgs',
                             max_iter=1000,
                             random_state=42)

log_reg.fit(X_train, y_train)

c:\Users\Iñigo Peña\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42)

In [6]:
from sklearn.metrics import accuracy_score, f1_score

y_pred = log_reg.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))

Accuracy: 0.9050387596899225
Macro-F1: 0.8245296524292908


 El **Accuracy** indica el porcentaje total de predicciones correctas, mientras que el **Macro-F1** promedia el F1-score de todas las clases sin ponderar por frecuencia, siendo especialmente útil cuando hay desbalance entre topics. El valor Macro-F1 no es muy lejano al Accuracy, por lo que el modelo no muestra dificultades con clases minoritarias, como podrian ser *"Informal / Conversational Lenguaje"* o *"Quantum Computing and Military Technology"* en nuestro caso.

In [7]:
from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = log_reg.predict(X_val)

acc = accuracy_score(y_val, y_val_pred)
f1_macro = f1_score(y_val, y_val_pred, average="macro")

print("\nClassification Report (Validation):")
print(classification_report(y_val, y_val_pred))

print("\nConfusion Matrix (Validation):")
print(confusion_matrix(y_val, y_val_pred))


Classification Report (Validation):
                                                             precision    recall  f1-score   support

Business Growth and Cloud Infrastructure in the AI Industry       0.92      0.91      0.91       159
              Financial and Market News and Corporate Sales       0.89      0.96      0.92       226
                         Informal / Conversational Lenguaje       1.00      0.50      0.67        18
                  Quantum Computing and Military Technology       1.00      0.67      0.80        12
                                   Stock Market and Trading       0.92      0.88      0.90       101

                                                   accuracy                           0.91       516
                                                  macro avg       0.94      0.78      0.84       516
                                               weighted avg       0.91      0.91      0.90       516


Confusion Matrix (Validation):
[[144  12   0   0  

El Classification Report muestra un rendimiento general alto (91% accuracy) y muy buenos resultados en las clases mayoritarias (*"Financial and Market News"*, *"Business Growth"*, con *F1 > 0.90*). Las clases minoritarias presentan mas problemas: *"Informal/Conversational Language"* alcanza solo 50% de recall y *"Quantum Computing"* un 67%. La Confusion Matrix indica que estas categorías se confunden sobre todo con *"Financial and Market News"*, lo que sugiere falta de datos o necesidad de técnicas de balanceo para mejorar su detección.

## Linear SVM

Implementamos ahora una Support Vector Machine (SVM) lineal, un clasificador robusto que busca el hiperplano óptimo que maximiza el margen entre clases. En espacios de alta dimensionalidad como el generado por TF-IDF (5000 features), los datos suelen ser linealmente separables, lo que puede hacer que SVM lineal sea particularmente efectivo. El parámetro C controla el trade-off entre maximizar el margen y minimizar errores de clasificación; valores más altos penalizan más los errores.

In [14]:
from sklearn.svm import LinearSVC

svm = LinearSVC(C=1.0)  # luego ajustar C
svm.fit(X_train, y_train)

LinearSVC()

In [15]:
pred = svm.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print("Macro-F1:", f1_score(y_test, pred, average="macro"))

Accuracy: 0.9282945736434108
Macro-F1: 0.8715570679691778


Los resultados de SVM muestran una mejora respecto a la Regresión Logística. Esto se explica por la naturaleza del problema: con la alta dimensionalidad de TF-IDF, las 5 clases se vuelven linealmente separables en este espacio de alta dimensión. SVM es especialmente bueno aprovechando esta separabilidad lineal al encontrar el hiperplano óptimo que maximiza el margen entre clases, lo que resulta en mejor generalización.

In [17]:
y_val_pred = svm.predict(X_val)

acc = accuracy_score(y_val, y_val_pred)
f1_macro = f1_score(y_val, y_val_pred, average="macro")

print("\nClassification Report (Validation):")
print(classification_report(y_val, y_val_pred))

print("\nConfusion Matrix (Validation):")
print(confusion_matrix(y_val, y_val_pred))


Classification Report (Validation):
                                                             precision    recall  f1-score   support

Business Growth and Cloud Infrastructure in the AI Industry       0.93      0.93      0.93       159
              Financial and Market News and Corporate Sales       0.92      0.94      0.93       226
                         Informal / Conversational Lenguaje       0.88      0.83      0.86        18
                  Quantum Computing and Military Technology       1.00      0.75      0.86        12
                                   Stock Market and Trading       0.93      0.90      0.91       101

                                                   accuracy                           0.92       516
                                                  macro avg       0.93      0.87      0.90       516
                                               weighted avg       0.92      0.92      0.92       516


Confusion Matrix (Validation):
[[148   8   0   0  